# Mask R-CNN — Pipeline webcam temps réel
**Affichage inline Jupyter** — compatible Wayland, pas de `cv2.imshow`

- Cellule 1 : imports
- Cellule 2 : classes + couleurs
- Cellule 3 : chargement modèle
- Cellule 4 : inférence + overlay
- Cellule 5 : **boucle webcam inline** ← c'est ici que tout se passe
- Cellule 6 : mode fichier vidéo (optionnel)

In [ ]:
import time
import cv2
import numpy as np
import os
import sys
from IPython.display import display, Image as IPImage, clear_output

import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2 import model_zoo

print("[OK] Imports OK")

In [ ]:
# ─────────────────────────────────────────────
#  Classes Cityscapes (8 classes d'instances)
# ─────────────────────────────────────────────
CITYSCAPES_CLASSES = [
    "person", "rider", "car", "truck", "bus", "train", "motorcycle", "bicycle"
]

CLASS_COLORS = {
    "person":       (60,  180, 255),
    "car":          (80,  220,  80),
    "bicycle":      (255, 140,  60),
    "motorcycle":   (200,  80, 200),
    "bus":          (60,  200, 200),
    "truck":        (100, 100, 220),
    "rider":        (20,   20, 220),
    "train":        (220, 180,  80),
}

print(f"[OK] {len(CITYSCAPES_CLASSES)} classes chargées : {CITYSCAPES_CLASSES}")

In [ ]:
# ─────────────────────────────────────────────
#  Chargement du modèle Detectron2
# ─────────────────────────────────────────────
import torch

CHECKPOINT_PATH = "../outputs/mask_rcnn_cityscapes/model_final.pth"  # ← adapte si besoin
SCORE_THRESH    = 0.5

def load_model(checkpoint_path: str, score_thresh: float = 0.5) -> DefaultPredictor:
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    ))
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CITYSCAPES_CLASSES)

    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"[INFO] Chargement du checkpoint : {checkpoint_path}")
        cfg.MODEL.WEIGHTS = checkpoint_path
    else:
        print(f"[WARN] Checkpoint non trouvé : {checkpoint_path}")
        print("[INFO] Fallback sur modèle COCO générique (80 classes, pour tester le pipeline)")
        cfg.MODEL.ROI_HEADS.NUM_CLASSES = 80
        cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
            "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
        )

    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] Device : {cfg.MODEL.DEVICE}")

    predictor = DefaultPredictor(cfg)
    print("[OK] Modèle prêt.")
    return predictor

predictor = load_model(CHECKPOINT_PATH, SCORE_THRESH)

In [ ]:
# ─────────────────────────────────────────────
#  Fonctions utilitaires : overlay + HUD
# ─────────────────────────────────────────────

def draw_results(
    frame: np.ndarray,
    instances,
    class_names: list,
    mask_alpha: float = 0.45,
    show_masks: bool = True,
    active_classes: set = None,
) -> tuple:
    overlay = frame.copy()
    stats = {}

    if not instances.has("pred_boxes") or len(instances) == 0:
        return overlay, stats

    boxes  = instances.pred_boxes.tensor.cpu().numpy()
    labels = instances.pred_classes.cpu().numpy()
    scores = instances.scores.cpu().numpy()
    masks  = instances.pred_masks.cpu().numpy() if instances.has("pred_masks") else None

    for i, (box, label, score) in enumerate(zip(boxes, labels, scores)):
        cls_name = class_names[label] if label < len(class_names) else f"class_{label}"

        if active_classes and cls_name not in active_classes:
            continue

        color = CLASS_COLORS.get(cls_name, (200, 200, 200))
        stats[cls_name] = stats.get(cls_name, 0) + 1

        # Masque
        if show_masks and masks is not None:
            bin_mask = masks[i].astype(bool)
            roi = overlay[bin_mask].astype(np.float32)
            if len(roi) > 0:
                blended = roi * (1.0 - mask_alpha) + np.array(color, dtype=np.float32) * mask_alpha
                overlay[bin_mask] = blended.astype(np.uint8)

        # Boîte
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)

        # Label
        label_txt = f"{cls_name} {score:.2f}"
        (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(overlay, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(overlay, label_txt, (x1 + 2, y1 - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

    return overlay, stats


def draw_hud(frame: np.ndarray, fps: float, stats: dict, show_masks: bool, frame_count: int) -> np.ndarray:
    lines = (
        [f"FPS : {fps:.1f}  |  Frame #{frame_count}",
         f"Masques : {'ON' if show_masks else 'OFF'}",
         "--- Détections ---"]
        + [f"  {cls}: {cnt}" for cls, cnt in sorted(stats.items())]
    )
    # Fond semi-transparent pour le HUD
    h_hud = 20 + len(lines) * 20
    roi = frame[0:h_hud, 0:220]
    dark = (roi.astype(np.float32) * 0.4).astype(np.uint8)
    frame[0:h_hud, 0:220] = dark

    for i, line in enumerate(lines):
        cv2.putText(frame, line, (8, 22 + i * 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1, cv2.LINE_AA)
    return frame


def frame_to_jpeg_bytes(frame_bgr: np.ndarray, max_width: int = 720) -> bytes:
    """Encode une frame BGR en JPEG pour affichage Jupyter."""
    h, w = frame_bgr.shape[:2]
    if w > max_width:
        scale = max_width / w
        frame_bgr = cv2.resize(frame_bgr, (max_width, int(h * scale)))
    ok, buf = cv2.imencode(".jpg", frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return buf.tobytes() if ok else b""

print("[OK] Fonctions utilitaires définies.")

In [ ]:
# ─────────────────────────────────────────────
#  Boucle webcam — affichage inline Jupyter
#  (compatible Wayland, pas de cv2.imshow)
# ─────────────────────────────────────────────

# ── Paramètres ──────────────────────────────
SOURCE       = 0        # 0 = webcam, ou "chemin/vers/video.mp4"
MAX_FRAMES   = 300      # Nombre max de frames avant arrêt automatique (None = infini)
SHOW_MASKS   = True     # Afficher les masques
RESIZE_W     = 640      # Largeur de traitement (None = taille native)
DISPLAY_FPS  = 5        # Rafraîchissement Jupyter (frames affichées / seconde réel)
                        # Augmenter = plus fluide mais plus de charge CPU
# ────────────────────────────────────────────

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    print(f"[ERREUR] Impossible d'ouvrir la source : {SOURCE}")
    print("Vérifie que ta webcam est connectée et accessible.")
else:
    print(f"[INFO] Webcam ouverte. Résolution native : "
          f"{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
    print("[INFO] Boucle démarrée. Interromps avec le bouton ⬛ (Stop) du notebook.")

    prev_time   = time.time()
    fps         = 0.0
    frame_count = 0
    last_display = time.time()
    display_handle = display(IPImage(b""), display_id=True)

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("[INFO] Fin du flux.")
                break

            if RESIZE_W is not None:
                h_orig, w_orig = frame.shape[:2]
                frame = cv2.resize(frame, (RESIZE_W, int(h_orig * RESIZE_W / w_orig)))

            # Inférence
            instances = predictor(frame)["instances"]

            # Overlay
            annotated, stats = draw_results(
                frame, instances,
                class_names=CITYSCAPES_CLASSES,
                mask_alpha=0.45,
                show_masks=SHOW_MASKS,
            )

            # FPS
            frame_count += 1
            now = time.time()
            fps = 0.9 * fps + 0.1 / max(now - prev_time, 1e-6)
            prev_time = now

            draw_hud(annotated, fps, stats, SHOW_MASKS, frame_count)

            # Affichage Jupyter (throttlé pour ne pas saturer le kernel)
            if (now - last_display) >= (1.0 / DISPLAY_FPS):
                jpeg = frame_to_jpeg_bytes(annotated)
                display_handle.update(IPImage(data=jpeg))
                last_display = now

            if MAX_FRAMES is not None and frame_count >= MAX_FRAMES:
                print(f"[INFO] {MAX_FRAMES} frames atteintes. Arrêt.")
                break

    except KeyboardInterrupt:
        print("[INFO] Interrompu.")
    finally:
        cap.release()
        print(f"[INFO] Terminé. {frame_count} frames traitées @ {fps:.1f} FPS moyen.")

In [ ]:
# ─────────────────────────────────────────────
#  Optionnel : inférence sur fichier vidéo
#  + sauvegarde de la vidéo annotée
# ─────────────────────────────────────────────

VIDEO_INPUT  = "chemin/vers/ta_video.mp4"   # ← modifie ici
VIDEO_OUTPUT = "output_annotated.mp4"
RESIZE_W_VID = 640

cap = cv2.VideoCapture(VIDEO_INPUT)
if not cap.isOpened():
    print(f"[ERREUR] Impossible d'ouvrir : {VIDEO_INPUT}")
else:
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_src = cap.get(cv2.CAP_PROP_FPS) or 25
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if RESIZE_W_VID:
        out_w, out_h = RESIZE_W_VID, int(h * RESIZE_W_VID / w)
    else:
        out_w, out_h = w, h

    writer = cv2.VideoWriter(
        VIDEO_OUTPUT,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps_src, (out_w, out_h)
    )

    display_handle = display(IPImage(b""), display_id=True)
    frame_count = 0
    t0 = time.time()

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if RESIZE_W_VID:
                frame = cv2.resize(frame, (out_w, out_h))

            instances = predictor(frame)["instances"]
            annotated, stats = draw_results(frame, instances, CITYSCAPES_CLASSES)

            fps_cur = frame_count / max(time.time() - t0, 1e-6)
            draw_hud(annotated, fps_cur, stats, True, frame_count)
            writer.write(annotated)
            frame_count += 1

            # Aperçu toutes les 10 frames
            if frame_count % 10 == 0:
                display_handle.update(IPImage(data=frame_to_jpeg_bytes(annotated)))
                print(f"\r  {frame_count}/{total} frames  ({fps_cur:.1f} FPS)", end="")

    except KeyboardInterrupt:
        print("\n[INFO] Interrompu.")
    finally:
        cap.release()
        writer.release()
        print(f"\n[OK] Vidéo sauvegardée : {VIDEO_OUTPUT}  ({frame_count} frames)")